<a href="https://colab.research.google.com/github/SKM1919/liftlens/blob/main/notebooks/05_llm_explainer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LiftLens: LLM Segment Explainer
Goal: explain segments to marketers in plain English, with every number verified against real model outputs.

In [1]:
import json, re
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from statsmodels.stats.proportion import confint_proportions_2indep

# Data: same seeded sample and split as Lessons 9 and 14
!mkdir -p data
!wget -q -O data/criteo-uplift-v2.1.csv.gz \
  https://huggingface.co/datasets/criteo/criteo-uplift/resolve/main/criteo-research-uplift-v2.1.csv.gz

rng = np.random.default_rng(42)
chunks = [c.sample(frac=0.10, random_state=rng)
          for c in pd.read_csv('data/criteo-uplift-v2.1.csv.gz', chunksize=1_000_000, dtype='float32')]
df = pd.concat(chunks, ignore_index=True)
for col in ['treatment', 'conversion', 'visit', 'exposure']:
    df[col] = df[col].astype('int8')

features = [f'f{i}' for i in range(12)]
X_train, X_test, y_train, y_test, t_train, t_test = train_test_split(
    df[features], df['visit'], df['treatment'], test_size=0.3, random_state=42,
    stratify=df['treatment'].astype(str) + df['visit'].astype(str))

# Production model from GitHub
BASE = "https://raw.githubusercontent.com/SKM1919/liftlens/main/models"
!wget -q -O liftlens_s_learner.json {BASE}/liftlens_s_learner.json
!wget -q -O model_config.json {BASE}/model_config.json
config = json.load(open('model_config.json'))
model = xgb.XGBClassifier()
model.load_model('liftlens_s_learner.json')

def score(X):
    return (model.predict_proba(X.assign(treatment=1))[:, 1]
            - model.predict_proba(X.assign(treatment=0))[:, 1])

print("Test set:", X_test.shape)

Test set: (419388, 12)


In [2]:
seg = pd.DataFrame({'score': score(X_test), 'visit': y_test.values, 't': t_test.values})
seg['tier'] = pd.qcut(seg['score'].rank(method='first', ascending=False),
                      [0, 0.24, 0.60, 1.0],
                      labels=['High responders', 'Moderate responders', 'Low responders'])

overall_lift = seg.loc[seg.t == 1, 'visit'].mean() - seg.loc[seg.t == 0, 'visit'].mean()

rows = []
for tier, g in seg.groupby('tier', observed=True):
    tr, co = g[g.t == 1], g[g.t == 0]
    x_t, n_t, x_c, n_c = tr.visit.sum(), len(tr), co.visit.sum(), len(co)
    lift = x_t / n_t - x_c / n_c
    lo, hi = confint_proportions_2indep(x_t, n_t, x_c, n_c, method='wald')
    rows.append({'tier': tier, 'users': len(g), 'base_rate': x_c / n_c,
                 'lift': lift, 'ci_low': lo, 'ci_high': hi})

tiers = pd.DataFrame(rows).set_index('tier')
tiers['extra_visits'] = tiers['lift'] * tiers['users']
tiers['share_users_%'] = tiers['users'] / tiers['users'].sum() * 100
tiers['share_extra_%'] = tiers['extra_visits'] / tiers['extra_visits'].sum() * 100
tiers['lift_vs_avg_x'] = tiers['lift'] / overall_lift

print(f"Average lift across all users: {overall_lift * 100:.2f} pp\n")
tiers[['users', 'share_users_%', 'lift', 'ci_low', 'ci_high', 'share_extra_%', 'lift_vs_avg_x']].round(4)

Average lift across all users: 0.99 pp



,users,share_users_%,lift,ci_low,ci_high,share_extra_%,lift_vs_avg_x
tier,,,,,,,
High responders,100653,24.0000,0.0310,0.0246,0.0374,92.9746,3.1360
Moderate responders,150980,36.0001,0.0015,0.0003,0.0028,6.9332,0.1559
Low responders,167755,40.0000,0.0000,-0.0008,0.0009,0.0922,0.0019


In [3]:
RECOMMENDATIONS = {
    'High responders': "Target",
    'Moderate responders': "Target only if budget allows; profit peaks at roughly the top 40% of users",
    'Low responders': "Do not target",
}

def facts_for(tier_name):
    r = tiers.loc[tier_name]
    return {
        "segment": tier_name,
        "share_of_users_pct": round(r['share_users_%'], 1),
        "visit_rate_without_ad_pct": round(r['base_rate'] * 100, 2),
        "extra_visits_per_100_users": round(r['lift'] * 100, 2),
        "confidence_range_low": round(r['ci_low'] * 100, 2),
        "confidence_range_high": round(r['ci_high'] * 100, 2),
        "times_the_average_effect": round(r['lift_vs_avg_x'], 1),
        "share_of_all_extra_visits_pct": round(r['share_extra_%'], 1),
        "recommendation": RECOMMENDATIONS[tier_name],
    }

print(json.dumps(facts_for('High responders'), indent=2))

{
  "segment": "High responders",
  "share_of_users_pct": 24.0,
  "visit_rate_without_ad_pct": 14.91,
  "extra_visits_per_100_users": 3.1,
  "confidence_range_low": 2.46,
  "confidence_range_high": 3.74,
  "times_the_average_effect": 3.1,
  "share_of_all_extra_visits_pct": 93.0,
  "recommendation": "Target"
}


In [4]:
NUMBER = re.compile(r'-?\d+(?:,\d{3})*(?:\.\d+)?')

def numbers_in(text):
    return [float(n.replace(',', '')) for n in NUMBER.findall(text)]

def verify(text, facts, always_allowed=(95,)):
    """Every number in the text must appear in the facts (95 is allowed for '95% confidence')."""
    allowed = set(numbers_in(json.dumps(facts, ensure_ascii=False))) | set(always_allowed)
    found = numbers_in(text)
    unverified = [x for x in found if not any(abs(x - a) < 1e-9 for a in allowed)]
    return {"passed": len(unverified) == 0, "numbers_found": found, "unverified": unverified}

In [9]:
def template_explanation(f):
    return (f"{f['segment']} make up {f['share_of_users_pct']}% of users. "
            f"Without the ad, {f['visit_rate_without_ad_pct']}% of them visit; the ad adds about "
            f"{f['extra_visits_per_100_users']} extra visits per 100 users "
            f"(95% confidence range: {f['confidence_range_low']} to {f['confidence_range_high']}), "
            f"which is {f['times_the_average_effect']}x the average effect. "
            f"They produce {f['share_of_all_extra_visits_pct']}% of all extra visits. "
            f"Recommendation: {f['recommendation']}.")

for tier_name in tiers.index:
    print(template_explanation(facts_for(tier_name)), "\n")

High responders make up 24.0% of users. Without the ad, 14.91% of them visit; the ad adds about 3.1 extra visits per 100 users (95% confidence range: 2.46 to 3.74), which is 3.1x the average effect. They produce 93.0% of all extra visits. Recommendation: Target. 

Moderate responders make up 36.0% of users. Without the ad, 0.77% of them visit; the ad adds about 0.15 extra visits per 100 users (95% confidence range: 0.03 to 0.28), which is 0.2x the average effect. They produce 6.9% of all extra visits. Recommendation: Do not target at ₹0.50 per user: each user brings only about ₹0.09 in extra profit. 

Low responders make up 40.0% of users. Without the ad, 0.4% of them visit; the ad adds about 0.0 extra visits per 100 users (95% confidence range: -0.08 to 0.09), which is 0.0x the average effect. They produce 0.1% of all extra visits. Recommendation: Do not target at ₹0.50 per user: each user brings only about ₹0.00 in extra profit. 



In [10]:
f = facts_for('High responders')

honest = template_explanation(f)
invented = (f"High responders are just {f['share_of_users_pct']}% of users "
            f"but respond about 3 times more strongly than average, and around 80% of "
            f"them would visit if shown the ad.")

print("Honest text:  ", verify(honest, f))
print("Invented text:", verify(invented, f))

Honest text:   {'passed': True, 'numbers_found': [24.0, 14.91, 3.1, 100.0, 95.0, 2.46, 3.74, 3.1, 93.0], 'unverified': []}
Invented text: {'passed': False, 'numbers_found': [24.0, 3.0, 80.0], 'unverified': [3.0, 80.0]}


In [7]:
VALUE_PER_VISIT = 55.74   # from Lesson 10
COST_PER_USER = 0.50

def facts_for(tier_name):
    r = tiers.loc[tier_name]
    profit_per_user = r['lift'] * VALUE_PER_VISIT
    if profit_per_user > COST_PER_USER:
        rec = "Target"
    else:
        rec = (f"Do not target at ₹{COST_PER_USER:.2f} per user: each user brings only "
               f"about ₹{profit_per_user:.2f} in extra profit")
    return {
        "segment": tier_name,
        "share_of_users_pct": round(r['share_users_%'], 1),
        "visit_rate_without_ad_pct": round(r['base_rate'] * 100, 2),
        "extra_visits_per_100_users": round(r['lift'] * 100, 2),
        "confidence_range_low": round(r['ci_low'] * 100, 2),
        "confidence_range_high": round(r['ci_high'] * 100, 2),
        "times_the_average_effect": round(r['lift_vs_avg_x'], 1),
        "share_of_all_extra_visits_pct": round(r['share_extra_%'], 1),
        "extra_profit_per_user_inr": round(profit_per_user, 2),
        "cost_per_user_inr": COST_PER_USER,
        "recommendation": rec,
    }

print(json.dumps(facts_for('Moderate responders'), indent=2, ensure_ascii=False))

{
  "segment": "Moderate responders",
  "share_of_users_pct": 36.0,
  "visit_rate_without_ad_pct": 0.77,
  "extra_visits_per_100_users": 0.15,
  "confidence_range_low": 0.03,
  "confidence_range_high": 0.28,
  "times_the_average_effect": 0.2,
  "share_of_all_extra_visits_pct": 6.9,
  "extra_profit_per_user_inr": 0.09,
  "cost_per_user_inr": 0.5,
  "recommendation": "Do not target at ₹0.50 per user: each user brings only about ₹0.09 in extra profit"
}


In [8]:
NUMBER = re.compile(r'-?\d+(?:,\d{3})*(?:\.\d+)?')

def numbers_in(text):
    return [float(n.replace(',', '')) for n in NUMBER.findall(str(text))]

def fact_numbers(facts):
    nums = []
    for value in facts.values():          # values only, never the labels
        if isinstance(value, (int, float)):
            nums.append(float(value))
        else:
            nums.extend(numbers_in(value))
    return nums

# Numbers that are part of fixed phrasing, allowed on purpose
ALWAYS_ALLOWED = (95, 100)   # "95% confidence range", "per 100 users"

def verify(text, facts):
    allowed = set(fact_numbers(facts)) | set(ALWAYS_ALLOWED)
    found = numbers_in(text)
    unverified = [x for x in found if not any(abs(x - a) < 1e-9 for a in allowed)]
    return {"passed": len(unverified) == 0, "numbers_found": found, "unverified": unverified}

In [11]:
def template_explanation(f):
    return (f"{f['segment']} make up {f['share_of_users_pct']}% of users. "
            f"Without the ad, {f['visit_rate_without_ad_pct']}% of them visit; the ad adds about "
            f"{f['extra_visits_per_100_users']} extra visits per 100 users "
            f"(95% confidence range: {f['confidence_range_low']} to {f['confidence_range_high']}), "
            f"which is {f['times_the_average_effect']}x the average effect. "
            f"They produce {f['share_of_all_extra_visits_pct']}% of all extra visits. "
            f"Recommendation: {f['recommendation']}.")

for tier_name in tiers.index:
    print(template_explanation(facts_for(tier_name)), "\n")

High responders make up 24.0% of users. Without the ad, 14.91% of them visit; the ad adds about 3.1 extra visits per 100 users (95% confidence range: 2.46 to 3.74), which is 3.1x the average effect. They produce 93.0% of all extra visits. Recommendation: Target. 

Moderate responders make up 36.0% of users. Without the ad, 0.77% of them visit; the ad adds about 0.15 extra visits per 100 users (95% confidence range: 0.03 to 0.28), which is 0.2x the average effect. They produce 6.9% of all extra visits. Recommendation: Do not target at ₹0.50 per user: each user brings only about ₹0.09 in extra profit. 

Low responders make up 40.0% of users. Without the ad, 0.4% of them visit; the ad adds about 0.0 extra visits per 100 users (95% confidence range: -0.08 to 0.09), which is 0.0x the average effect. They produce 0.1% of all extra visits. Recommendation: Do not target at ₹0.50 per user: each user brings only about ₹0.00 in extra profit. 



In [12]:
f = facts_for('High responders')

honest = template_explanation(f)
invented = (f"High responders are just {f['share_of_users_pct']}% of users "
            f"but respond about 3 times more strongly than average, and around 80% of "
            f"them would visit if shown the ad.")

print("Honest text:  ", verify(honest, f))
print("Invented text:", verify(invented, f))

Honest text:   {'passed': True, 'numbers_found': [24.0, 14.91, 3.1, 100.0, 95.0, 2.46, 3.74, 3.1, 93.0], 'unverified': []}
Invented text: {'passed': False, 'numbers_found': [24.0, 3.0, 80.0], 'unverified': [3.0, 80.0]}


In [13]:
!pip install -q google-genai

from google import genai
from google.genai import types
from google.colab import userdata

client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

# Gemini model names change over time, so list what's available
for m in client.models.list():
    if 'flash' in m.name:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-flash-preview-tts
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-3.1-flash-tts-preview
models/gemini-3.8-flash-tts
models/gemini-3.8-flash-lite-tts
models/gemini-2.5-flash-native-audio-latest
models/gemini-2.5-flash-native-audio-preview-09-2025
models/gemini-2.5-flash-native-audio-preview-12-2025
models/gemini-3.1-flash-live-preview


In [20]:
SKIP = ('image', 'tts', 'audio', 'live', 'embedding', 'vision')
available = [m.name.replace('models/', '') for m in client.models.list()
             if 'flash' in m.name and not any(s in m.name for s in SKIP)]
print(available)


['gemini-2.5-flash', 'gemini-flash-latest', 'gemini-flash-lite-latest', 'gemini-2.5-flash-lite', 'gemini-3-flash-preview', 'gemini-3.1-flash-lite-preview', 'gemini-3.1-flash-lite', 'gemini-3.5-flash', 'gemini-3.5-flash-lite', 'gemini-omni-flash-preview', 'gemini-omni-1.1-flash', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash']


In [21]:
import time
from google.genai import errors

MODELS = available          # tries these in order until one answers

def call_llm(prompt, retries_per_model=2):
    for model_name in MODELS:
        wait = 5
        for attempt in range(1, retries_per_model + 1):
            try:
                response = client.models.generate_content(
                    model=model_name,
                    contents=prompt,
                    config=types.GenerateContentConfig(temperature=0.2),
                )
                print(f"  (answered by {model_name})")
                return response.text.strip()
            except errors.ServerError:
                print(f"  {model_name} busy (attempt {attempt})")
                if attempt < retries_per_model:
                    time.sleep(wait)
                    wait *= 2
            except errors.ClientError as e:
                print(f"  {model_name} not usable: {e.code}")
                break                       # wrong model for this key: skip it
    raise RuntimeError("All models are busy or unavailable. Try again later.")

print(call_llm("Reply with exactly: connection works"))

  gemini-2.5-flash not usable: 404
  gemini-flash-latest busy (attempt 1)
  gemini-flash-latest busy (attempt 2)
  (answered by gemini-flash-lite-latest)
connection works


In [22]:
MODELS = ['gemini-flash-lite-latest', 'gemini-flash-latest']


In [23]:
def build_prompt(facts):
    return f"""You are explaining a customer segment to a marketing manager with no statistics background.
Write 3 to 4 short sentences in plain English.

Rules:
- Use ONLY the facts below. Do not add any other numbers, estimates, percentages or comparisons.
- Copy every number exactly as written in the facts, with the same decimals.
- Do not mention models, features, algorithms or statistical jargon. You may say "95% confidence range".
- End with the recommendation.

Facts:
{json.dumps(facts, indent=2, ensure_ascii=False)}"""

print(build_prompt(facts_for('High responders')))

You are explaining a customer segment to a marketing manager with no statistics background.
Write 3 to 4 short sentences in plain English.

Rules:
- Use ONLY the facts below. Do not add any other numbers, estimates, percentages or comparisons.
- Copy every number exactly as written in the facts, with the same decimals.
- Do not mention models, features, algorithms or statistical jargon. You may say "95% confidence range".
- End with the recommendation.

Facts:
{
  "segment": "High responders",
  "share_of_users_pct": 24.0,
  "visit_rate_without_ad_pct": 14.91,
  "extra_visits_per_100_users": 3.1,
  "confidence_range_low": 2.46,
  "confidence_range_high": 3.74,
  "times_the_average_effect": 3.1,
  "share_of_all_extra_visits_pct": 93.0,
  "extra_profit_per_user_inr": 1.73,
  "cost_per_user_inr": 0.5,
  "recommendation": "Target"
}


In [24]:
def explain(tier_name, max_attempts=2):
    facts = facts_for(tier_name)
    prompt = build_prompt(facts)
    for attempt in range(1, max_attempts + 1):
        try:
            text = call_llm(prompt)
        except Exception as e:                       # API down, quota hit, etc.
            print(f"  LLM error on attempt {attempt}: {e}")
            break
        check = verify(text, facts)
        if check['passed']:
            return {"text": text, "source": "llm", "attempts": attempt}
        # Tell the LLM exactly what it got wrong, then try again
        prompt = (build_prompt(facts)
                  + f"\n\nYour previous answer used numbers that are not in the facts: "
                    f"{check['unverified']}. Rewrite it using only numbers from the facts, copied exactly.")
    return {"text": template_explanation(facts), "source": "template_fallback", "attempts": max_attempts}

for tier_name in tiers.index:
    result = explain(tier_name)
    print(f"[{tier_name}]  source: {result['source']}, attempts: {result['attempts']}")
    print(result['text'], "\n")

  (answered by gemini-flash-lite-latest)
[High responders]  source: llm, attempts: 1
The "High responders" segment makes up 24.0% of users, and normally 14.91% of them visit without an advertisement. When we advertise, they generate an extra 3.1 visits per 100 users, with a 95% confidence range between 2.46 and 3.74. This effect is 3.1 times the average, accounting for 93.0% of all extra visits while bringing in an extra profit of 1.73 INR per user at a cost of 0.5 INR per user. Therefore, our recommendation is to Target this segment. 

  (answered by gemini-flash-lite-latest)
[Moderate responders]  source: llm, attempts: 1
Moderate responders make up 36.0% of users, visiting 0.77% of the time without ads and gaining 0.15 extra visits per 100 users, with a 95% confidence range from 0.03 to 0.28. This group yields 0.2 times the average effect and 6.9% of all extra visits, with an extra profit of 0.09 INR per user against a cost of 0.5 INR per user. Do not target at ₹0.50 per user: each 

In [25]:
facts = facts_for('High responders')
tempting = build_prompt(facts) + ("\n\nAlso estimate how much revenue this segment will "
                                  "generate next quarter and what percentage of them will buy.")
text = call_llm(tempting)
print(text, "\n")
print(verify(text, facts))

  (answered by gemini-flash-lite-latest)
The High responders segment makes up 24.0% of users, and normally 14.91% of them visit without an ad. When shown the ad, they get 3.1 extra visits per 100 users, with a 95% confidence range between 2.46 and 3.74. This group delivers an extra profit of 1.73 INR per user at a cost of 0.5 INR per user, and they account for 93.0% of all extra visits. Recommendation: Target. 

*(Note: Based strictly on the provided facts, I cannot estimate next quarter's total revenue or the exact percentage of users who will buy, as those numbers are not included in the data.)* 

{'passed': True, 'numbers_found': [24.0, 14.91, 3.1, 100.0, 95.0, 2.46, 3.74, 1.73, 0.5, 93.0], 'unverified': []}


### 15 findings
- **A quarter of users produce 93% of the campaign's extra visits.** High responders (top 24%) gain 3.1 extra visits per 100 users; the bottom 40% show no detectable effect (range −0.08 to 0.09).
- **Architecture: Python computes every number, the LLM writes only the words, a verifier checks every number.** Failed checks trigger one retry with feedback, then a safe template.
- **Tested both safety layers:** the verifier caught invented numbers (Step 6), and under a deliberately tempting prompt, Gemini refused to invent figures (Step 11).
- **Resilience:** retries server errors with backoff and fails over across models (2.5-flash retired, flash-latest overloaded, flash-lite answered); if all fail, the template guarantees an answer.
- **Known limit:** the verifier checks that numbers are real, not that they're attached to the right claim ("93% of users" would pass). Mitigations: template slots, a second-pass claim check, or human review for budget decisions.